## defines

In [1]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from pymongo.errors import OperationFailure
from pprint import pprint
from dotenv import load_dotenv
import os, time
from langchain_core.tools import tool
from typing import List
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import ToolMessage
from langgraph.graph import END, StateGraph, START
import voyageai
from langgraph.checkpoint.mongodb import MongoDBSaver


load_dotenv()
print("MONGO_URI loaded:", bool(os.getenv("MONGO_URI")))
print("GOOGLE_API_KEY loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("VOYAGEAI_API_KEY loaded:", bool(os.getenv("VOYAGEAI_API_KEY")))

MONGO_URI loaded: True
GOOGLE_API_KEY loaded: True
VOYAGEAI_API_KEY loaded: True


C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### connect to mongo DB

In [2]:
def init_mongodb():
    """
    Initialize MongoDB client and collections.

    Returns:
        tuple: MongoDB client, vector search collection, full documents collection.
    """
    mongodb_client = MongoClient(os.getenv("MONGO_URI"))
    
    DB_NAME = "ai_agents"
    
    vs_collection = mongodb_client[DB_NAME]["chunked_docs"]
    
    full_collection = mongodb_client[DB_NAME]["full_docs"]
    
    return mongodb_client, vs_collection, full_collection


### graph

In [3]:
# Define the graph state type with messages that can accumulate
class GraphState(TypedDict):
    # Define a messages field that keeps track of conversation history
    messages: Annotated[list, add_messages]

### generate embedding

In [4]:
def generate_embedding(text: str) -> List[float]:
    """
    Generate embedding for a piece of text.

    Args:
        text (str): The text to embed.
        embedding_model (voyage-4): The embedding model.

    Returns:
        List[float]: The embedding of the text.
    """

    embedding_model = voyageai.Client(api_key=os.getenv("VOYAGEAI_API_KEY"))

    embedding = embedding_model.embed(text, model="voyage-4", input_type="query").embeddings[0]
    
    return embedding


## tool: get_information_for_question_answering

In [5]:
@tool 
def get_information_for_question_answering(user_query: str) -> str:
    """
    Retrieve relevant documents for a user query using vector search.

    Args:
        user_query (str): The user's query.

    Returns:
        str: The retrieved documents as a string.
    """

    query_embedding = generate_embedding(user_query)

    vs_collection = init_mongodb()[1]
    
    pipeline = [
        {
            # Use vector search to find similar documents
            "$vectorSearch": {
                "index": "vector_index",  # Name of the vector index
                "path": "embedding",       # Field containing the embeddings
                "queryVector": query_embedding,  # The query embedding to compare against
                "numCandidates": 150,      # Consider 150 candidates (wider search)
                "limit": 5,                # Return only top 5 matches
            }
        },
        {
            # Project only the fields we need
            "$project": {
                "_id": 0,                  # Exclude document ID
                "body": 1,                 # Include the document body
                "score": {"$meta": "vectorSearchScore"},  # Include the similarity score
            }
        },
    ]
    
    results = vs_collection.aggregate(pipeline)
    
    context = "\n\n".join([doc.get("body") for doc in results])
    
    return context


## tool: get_page_content_for_summarization

In [6]:
@tool 
def get_page_content_for_summarization(user_query: str) -> str:
    """
    Retrieve the content of a documentation page for summarization.

    Args:
        user_query (str): The user's query (title of the documentation page).

    Returns:
        str: The content of the documentation page.
    """
    full_collection = init_mongodb()[2]

    query = {"title": user_query}
    
    projection = {"_id": 0, "body": 1}
    
    document = full_collection.find_one(query, projection)
    
    if document:
        return document["body"]
    else:
        return "Document not found"

### agent node

In [7]:
def agent(state: GraphState, llm_with_tools) -> GraphState:
    """
    Agent node.

    Args:
        state (GraphState): The graph state.
        llm_with_tools: The LLM with tools.

    Returns:
        GraphState: The updated messages.
    """

    messages = state["messages"]
    
    result = llm_with_tools.invoke(messages)
    
    return {"messages": [result]}

### Tool node

In [8]:
def tool_node(state: GraphState, tools_by_name) -> GraphState:
    """
    Tool node.

    Args:
        state (GraphState): The graph state.
        tools_by_name (Dict[str, Callable]): The tools by name.

    Returns:
        GraphState: The updated messages.
    """
    result = []
    
    tool_calls = state["messages"][-1].tool_calls
    
    for tool_call in tool_calls:
        tool = tools_by_name[tool_call["name"]]
        
        observation = tool.invoke(tool_call["args"])
        
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    
    return {"messages": result}

### route tools

In [9]:
def route_tools(state: GraphState):
    """
    Route to the tool node if the last message has tool calls. Otherwise, route to the end.

    Args:
        state (GraphState): The graph state.

    Returns:
        str: The next node to route to.
    """
    messages = state.get("messages", [])
    
    if len(messages) > 0:
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    
    return END


## Initialize the graph

In [10]:
def init_graph(llm_with_tools, tools_by_name, mongodb_client):
    """
    Initialize the graph.

    Args:
        llm_with_tools: The LLM with tools.
        tools_by_name (Dict[str, Callable]): The tools by name.
        mongodb_client (MongoClient): The MongoDB client.

    Returns:
        StateGraph: The compiled graph.
    """
    graph = StateGraph(GraphState)
    
    graph.add_node("agent", lambda state: agent(state, llm_with_tools))
    
    graph.add_node("tools", lambda state: tool_node(state, tools_by_name))
    
    graph.add_edge(START, "agent")
    
    graph.add_edge("tools", "agent")
    
    graph.add_conditional_edges("agent", route_tools, {"tools": "tools", END: END})

    checkpointer = MongoDBSaver(mongodb_client)
    
    return graph.compile(checkpointer=checkpointer)

### execute the graph

In [11]:
def execute_graph(app, thread_id: str, user_input: str) -> None:
    """
    Stream outputs from the graph.

    Args:
        app: The compiled graph application.
        thread_id (str): The thread ID.
        user_input (str): The user's input.
    """
    input = {"messages": [("user", user_input)]}
    
    config = {"configurable": {"thread_id": thread_id}}
    
    for output in app.stream(input, config):
        for key, value in output.items():
            print(f"Node {key}:")
            print(value)
    
    print("---FINAL ANSWER---")
    
    print(value["messages"][-1].content)


## our main function

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

def main():
    """
    Main function to initialize and execute the graph.
    """
    mongodb_client, vs_collection, full_collection = init_mongodb()
    
    tools = [
        get_information_for_question_answering,
        get_page_content_for_summarization
    ]
    
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.1,
        api_key=os.getenv("GOOGLE_API_KEY")
    )    
    
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "You are a helpful AI assistant."
                " You are provided with tools to answer questions and summarize technical documentation related to MongoDB."
                " Think step-by-step and use these tools to get the information required to answer the user query."
                " Do not re-run tools unless absolutely necessary."
                " If you are not able to get enough information using the tools, reply with I DON'T KNOW."
                " You have access to the following tools: {tool_names}."
            ),
            MessagesPlaceholder(variable_name="messages"),
        ]
    )
    
    prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))
    
    bind_tools = llm.bind_tools(tools)
    
    llm_with_tools = prompt | bind_tools
    
    tools_by_name = {tool.name: tool for tool in tools}
    
    app = init_graph(llm_with_tools, tools_by_name, mongodb_client)
    
    execute_graph(app, "1" ,"What are some best practices for data backups in MongoDB?")
    
    execute_graph(app, "1", "What did I just ask you?")
    

# invoke main

In [13]:
main()

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_information_for_question_answering', 'arguments': '{"user_query": "best practices for data backups in MongoDB"}'}, '__gemini_function_call_thought_signatures__': {'call_88037': 'EoQDCoEDAWkUfRMasb5Jmvj6fjUNEALsaHMzJ2oQI13e01LW/r9zVnz7hN7kj9Zz25lcG3OM9ukKrfOcTYKvBz+OhGgHW0SPSYwTmKv8QRL1H8+ImBcO4H1G2VKE1bJnyw74ItWAlyYeD2v2MR2m7xezL4OjFhYua5JS+gM2g7DPtNZCz6DSIyLhQSfGHICXRSSGBjXd0nxLLbE9lSYZ3IKMLy2MNPVZCC7BDI66VIOLnLGOKPXhJNEMq5dVfbipgprLyLLNnjHQfmezfOrymkneWYCYCjnvyfz2G2vPVz0m2pWKyWb5LdFZiKK4RomY5R2D1cQwdnTKHO/D08PByoK8/EYxmuIvzgheV3Turt8V+pdyPE9lCxVkLjf2q5Pzp+aZEMSFyZy7JwOIj0Ulyjppxndmy6TXpubHAEpj+QwdUJylkk5dEx7t37Y4W0zSxNuvQQ6enkmN4HsuPtEtnq7rYYVeq6j0fogyS5qJKKbhiqpKi1q2SH+2wkVUX0tKiv3OZNxN8w=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1a5e6-505a-7a00-b4f7-0fc1797cbf34-0', tool_ca

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_information_for_question_answering', 'arguments': '{"user_query": "MongoDB backup strategies methods best practices"}'}, '__gemini_function_call_thought_signatures__': {'call_1853': 'EtUBCtIBAWkUfRPEsbQFounGAMSodwhLRpZ2RE2WRlKIMDNf9HgUIcSP5nf29VAJQYBU4JbaYQXAsRC+h+RpTipXlSCrXmmBrYm6fCx9+o7qvpN77C53qM3J+OaZvBRPZgKKas8ljaEIoLJLTHIryxfN5A/yJPJxQJjoDggHMug0ikyLlUkn1Z13OYTyqAoUJH2/qldfodGpsUQAiEyGjQQQsSjWU3IB/K2b6i1uZWBvKN5sPzi33w0d6wRgA3puK8h5SwwJOjhUxk15UYq5ISXJ4FmZ/lOY'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1a5e6-6c90-70d1-987c-d994126b38ab-0', tool_calls=[{'name': 'get_information_for_question_answering', 'args': {'user_query': 'MongoDB backup strategies methods best practices'}, 'id': 'call_1853', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 10

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_information_for_question_answering', 'arguments': '{"user_query": "MongoDB Backup Methods Atlas Cloud Manager Ops Manager file system snapshots mongodump"}'}, '__gemini_function_call_thought_signatures__': {'call_113772': 'ErABCq0BAWkUfROD5tf+gyFp4oueSM7PmQ4PDPlXmYoCQrvx/pdBRugVs8xYgyXgD6floCFJ9rxcO97l87rnlkDL9yE9JPscOSzudgiyXT5affbxdu/AuFR03+9payVoy7QFsohUdYyYWheNW+W+8Ia9l4wsF9mq9tU8/GqElLeaIDDOzNaLVTmazwr/dzxffB9lEsY8s48rySrHFJ0u7NRCd/sEBjKZMkFHI+BB4ZyBeaw='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1a5e6-87ba-7331-a6e4-42f68727c8ca-0', tool_calls=[{'name': 'get_information_for_question_answering', 'args': {'user_query': 'MongoDB Backup Methods Atlas Cloud Manager Ops Manager file system snapshots mongodump'}, 'id': 'call_113772', 'type': 'tool_call'}], invalid_tool_calls=[], us

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[{'type': 'text', 'text': 'Based on MongoDB documentation, here are key best practices and methods for data backups:\n\n### 1. Use Coordinated Backup Services / Management Tools\nFor managed deployments and sharded clusters, use enterprise tools that support coordinated backup and restore processes. These preserve the atomicity guarantees of transactions across shards:\n* **MongoDB Atlas**\n* **MongoDB Cloud Manager**\n* **MongoDB Ops Manager**\n\n---\n\n### 2. File System Snapshots\n* Use **file system snapshots** to back up each component individually (such as in a sharded cluster).\n* If supported by your system configuration, file system snapshot backups can be more efficient than logical backup tools.\n\n---\n\n### 3. Using `mongodump` and `mongorestore`\n* Use `mongodump` to back up components individually for smaller deployments or logical backups.\n* **Stop or window the cluster balancer:** When running backups on sharded clusters, st

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[{'type': 'text', 'text': 'You just asked: **"What are some best practices for data backups in MongoDB?"**', 'extras': {'signature': 'Es4CCssCAWkUfRMG1QUDO4NpPn1TDUUtdAxazWcByBXCVNOwgUMlKF7MwbZBteUawVTvOf/c6TaVENTYQuqVbct27MwSg3q+rpykfU3MR901tbRn6ZjlnrOCPHSTnJvFgyrVdIyi8utZBZtIoudcZfdZmvntytqUNz1xQbcTiRGICO20jyMRIy4psuD8MzBWDkS8mEthIg55St5zXLqfc8zLLieCZH5C1jaZNtNay6T83+NtObOSzYGlp4QACPN7d/JSM4vjk7ygoQG2M7MRwzUnqT/WUwIxQvq3YlcvRNNsWF8Psu6Q+u+owZJUbBrcsL+E9AWmFYprhbtPHJw2MmXh0Y2okd1Yg2BH8eNH34ajp3/V5/rxulyS209l2H0+Iin8ja7jUBaPUPgl9SWgu7Meg9DawxHVstqqfRseciJgSGj9nWBCJ1Fn+J6pJ7mdGQ=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1a5e6-f15b-7da0-8a2a-59fc7e607c2b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2875, 'output_tokens': 68, 'total_tokens': 2943, 'input_token_details': {'cache_read